# Lesson 9a: Proximal Policy Optimization (PPO) - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand Trust Region Methods** - Constrained policy optimization
2. **Master TRPO** - Trust Region Policy Optimization foundation
3. **Learn PPO-Clip** - Clipped surrogate objective
4. **Understand PPO-Penalty** - Adaptive KL penalty
5. **Master Implementation Details** - Critical tricks for success
6. **Learn Why PPO Dominates** - Industry standard 2025
7. **Understand Variants** - PPO extensions and improvements

PPO is the most widely used deep RL algorithm in 2025, powering ChatGPT, robotics, and more.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. The Problem with Vanilla Policy Gradients

### Instability Issues

**Policy gradient update** (Actor-Critic, Lesson 8):
$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

**Problems**:

**1. Step size sensitivity**:
- Too small → slow learning
- Too large → policy collapse
- Hard to tune learning rate

**2. Sample inefficiency**:
- On-policy: must throw away data after update
- Cannot reuse old trajectories
- Expensive environment interactions

**3. Training instability**:
- Performance can collapse suddenly
- Hard to recover from bad updates
- Non-monotonic improvement

**4. Destructive updates**:
- One bad update can ruin policy
- Gradient estimator has high variance
- No safety guarantees

### Why This Happens

**Policy gradient is unconstrained**:
- Can take arbitrarily large steps
- New policy $\pi_{\theta'}$ can be very different from $\pi_\theta$
- Changes in policy distribution not controlled

**Solution**: Constrain policy updates!

### The Trust Region Idea

**Key insight**: Limit how much policy can change

**Trust region**: Region where we "trust" our approximation

**Constrained optimization**:
$$\max_\theta J(\theta) \quad \text{subject to} \quad D(\pi_\theta, \pi_{\theta_{\text{old}}}) \leq \delta$$

where $D$ is distance/divergence between policies

**Benefits**:
- ✅ Stable updates (monotonic improvement)
- ✅ Better sample efficiency
- ✅ Prevents policy collapse
- ✅ More robust to hyperparameters

## 2. TRPO: Trust Region Policy Optimization

### KL Divergence Constraint

**Measure policy distance**: KL divergence

$$D_{KL}(\pi_{\theta_{\text{old}}} \| \pi_\theta) = \mathbb{E}_{s \sim \rho_{\theta_{\text{old}}}} \left[ \sum_a \pi_{\theta_{\text{old}}}(a|s) \log \frac{\pi_{\theta_{\text{old}}}(a|s)}{\pi_\theta(a|s)} \right]$$

**Properties**:
- KL ≥ 0 (equality iff policies identical)
- Measures how much information lost using $\pi_\theta$ instead of $\pi_{\theta_{\text{old}}}$
- Smooth, differentiable

### TRPO Objective

**Maximize expected advantage**:
$$\max_\theta \mathbb{E}_{s,a \sim \pi_{\theta_{\text{old}}}} \left[ \frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} A^{\pi_{\theta_{\text{old}}}}(s,a) \right]$$

**Subject to constraint**:
$$\mathbb{E}_{s \sim \pi_{\theta_{\text{old}}}} [D_{KL}(\pi_{\theta_{\text{old}}}(\cdot|s) \| \pi_\theta(\cdot|s))] \leq \delta$$

### Importance Sampling

**Probability ratio**:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

**Objective becomes**:
$$L^{CPI}(\theta) = \mathbb{E}_t [r_t(\theta) A_t]$$

where CPI = Conservative Policy Iteration

**Why importance sampling?**
- Use data from old policy $\pi_{\theta_{\text{old}}}$
- Evaluate under new policy $\pi_\theta$
- Reuse samples (better sample efficiency!)

### TRPO Algorithm

```
Initialize policy πθ

For iteration = 1, 2, 3, ...:
  # Collect trajectories
  Run policy πθ_old to collect D = {(s,a,r,s')}
  
  # Estimate advantages
  Compute A^π(s,a) for all (s,a) in D
  
  # Solve constrained optimization
  θ ← argmax_θ E[r(θ) A]  subject to  E[KL] ≤ δ
  
  # Use conjugate gradient + line search
  (Complex second-order optimization)
```

### Theoretical Guarantee

**Theorem** (Schulman et al., 2015):

TRPO guarantees monotonic improvement:
$$J(\pi_{\text{new}}) \geq J(\pi_{\text{old}})$$

**Proof sketch**: KL constraint ensures new policy stays close enough that advantage estimates remain valid.

### TRPO Limitations

**Cons**:
- ❌ Complex implementation (conjugate gradient, line search)
- ❌ Computationally expensive (second-order optimization)
- ❌ Hard to scale to large networks
- ❌ Difficult to debug

**Pros**:
- ✅ Monotonic improvement guarantee
- ✅ Sample efficient
- ✅ Stable training

**Question**: Can we get TRPO benefits with simpler algorithm?

**Answer**: YES → PPO!

## 3. PPO: Proximal Policy Optimization

### The PPO Breakthrough

**Key insight** (Schulman et al., 2017):

Instead of hard KL constraint, use **clipped objective**!

**Advantages over TRPO**:
- ✅ First-order optimization (no conjugate gradient)
- ✅ Simple to implement
- ✅ Scales to large networks
- ✅ Similar performance to TRPO
- ✅ More sample efficient than vanilla PG

### PPO-Clip Objective

**Probability ratio**:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

**Clipped surrogate objective**:
$$L^{CLIP}(\theta) = \mathbb{E}_t [\min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t)]$$

where $\epsilon$ is clip parameter (typically 0.1-0.2)

**Clipping function**:
$$\text{clip}(r_t, 1-\epsilon, 1+\epsilon) = \begin{cases}
1-\epsilon & \text{if } r_t < 1-\epsilon \\
r_t & \text{if } 1-\epsilon \leq r_t \leq 1+\epsilon \\
1+\epsilon & \text{if } r_t > 1+\epsilon
\end{cases}$$

### Understanding the Clipping

**Case 1**: Positive advantage ($A_t > 0$)
- Want to increase $\pi_\theta(a|s)$
- But clip prevents $r_t$ from going above $1+\epsilon$
- Limits how much we can increase probability

**Case 2**: Negative advantage ($A_t < 0$)
- Want to decrease $\pi_\theta(a|s)$
- But clip prevents $r_t$ from going below $1-\epsilon$
- Limits how much we can decrease probability

**Effect**: Conservative updates, prevents large policy changes

### Visualization of Clipping

**For positive advantage** ($A > 0$):
```
Objective
    |
    |        _____________  ← Clipped at (1+ε)A
    |      /
    |    /  
    |  /    
    |/______________________
  1-ε     1      1+ε         r (ratio)
```

**Interpretation**:
- Increasing probability is limited
- No incentive to increase beyond clip
- Prevents over-optimization

### PPO-Penalty (Alternative)

**Add KL penalty instead of clipping**:

$$L^{KLPEN}(\theta) = \mathbb{E}_t [r_t(\theta) A_t - \beta D_{KL}(\pi_{\theta_{\text{old}}} \| \pi_\theta)]$$

where $\beta$ is adaptive coefficient:
- If KL too high: increase $\beta$
- If KL too low: decrease $\beta$

**PPO-Clip is preferred** (simpler, no $\beta$ tuning)

### Complete PPO Objective

**Full loss function**:
$$L(\theta) = L^{CLIP}(\theta) - c_1 L^{VF}(\theta) + c_2 H(\pi_\theta)$$

where:
- $L^{CLIP}$: Clipped policy objective
- $L^{VF} = (V_\theta(s) - V^{\text{target}})^2$: Value function loss
- $H(\pi_\theta) = -\sum_a \pi(a|s) \log \pi(a|s)$: Entropy bonus
- $c_1, c_2$: Loss coefficients (typical: $c_1=0.5$, $c_2=0.01$)

**Why these terms?**
- Value function: Better advantage estimates (actor-critic)
- Entropy: Encourages exploration
- Combined: Single network, joint optimization

## 4. PPO Algorithm

### Complete Algorithm

```
Initialize:
  - Policy network πθ (actor)
  - Value network Vφ (critic)
  - Or shared network with two heads

Hyperparameters:
  - ε: clip parameter (0.1-0.2)
  - K: epochs per update (3-10)
  - M: minibatch size (64-256)
  - T: timesteps per batch (2048-4096)

For iteration = 1, 2, 3, ...:
  
  # 1. Collect trajectories
  Run policy πθ_old for T timesteps
  Collect D = {(s_t, a_t, r_t, s_{t+1})}
  
  # 2. Compute advantages
  For each timestep t:
    Compute A^GAE_t using GAE(λ)
    Compute returns G_t = A_t + V(s_t)
  
  # 3. PPO update
  For epoch = 1 to K:
    # Shuffle data
    Shuffle D into random minibatches
    
    For each minibatch:
      # Compute ratios
      r(θ) = π_θ(a|s) / π_θ_old(a|s)
      
      # Clipped objective
      L^CLIP = E[min(r·A, clip(r, 1-ε, 1+ε)·A)]
      
      # Value loss
      L^VF = E[(V_φ(s) - G)²]
      
      # Entropy bonus
      H = E[-Σ π(a|s) log π(a|s)]
      
      # Total loss
      L = -L^CLIP + c₁·L^VF - c₂·H
      
      # Gradient descent
      θ, φ ← θ, φ - α∇L
  
  # 4. Update old policy
  θ_old ← θ
```

### Key Implementation Details

**1. Advantage estimation**: Use GAE(λ)
$$A_t^{\text{GAE}(\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$

**2. Advantage normalization**:
```python
A = (A - A.mean()) / (A.std() + 1e-8)
```
Critical for stability!

**3. Value function clipping** (optional):
```python
V_clipped = V_old + clip(V - V_old, -ε, ε)
L_VF = max((V - G)², (V_clipped - G)²)
```

**4. Learning rate annealing**:
- Start: 3e-4
- Anneal linearly to 0
- Improves final performance

**5. Gradient clipping**:
```python
torch.nn.utils.clip_grad_norm_(params, max_norm=0.5)
```

**6. Multiple epochs**: K=3-10 epochs per batch
- Reuse collected data K times
- Much better sample efficiency than vanilla PG
- But not too many (overfitting to batch)

### PPO Hyperparameters

**Standard values** (from OpenAI baselines):
```python
clip_epsilon = 0.2
learning_rate = 3e-4
n_steps = 2048
batch_size = 64
n_epochs = 10
gamma = 0.99
gae_lambda = 0.95
value_coef = 0.5
entropy_coef = 0.01
max_grad_norm = 0.5
```

**These work well across many tasks!**

## 5. Why PPO Dominates

### Comparison with Other Algorithms

| Algorithm | Sample Efficiency | Stability | Simplicity | Performance |
|-----------|-------------------|-----------|------------|-------------|
| REINFORCE | ★☆☆☆☆ | ★★☆☆☆ | ★★★★★ | ★★☆☆☆ |
| A2C/A3C | ★★☆☆☆ | ★★★☆☆ | ★★★★☆ | ★★★☆☆ |
| TRPO | ★★★★☆ | ★★★★★ | ★★☆☆☆ | ★★★★☆ |
| **PPO** | **★★★★☆** | **★★★★★** | **★★★★☆** | **★★★★★** |
| DQN | ★★★★★ | ★★★☆☆ | ★★★☆☆ | ★★★★☆ |
| SAC | ★★★★★ | ★★★★☆ | ★★★☆☆ | ★★★★★ |

**PPO sweet spot**: Best balance of all factors!

### Why Industry Loves PPO

**1. Reliability**:
- Works out of the box
- Robust to hyperparameters
- Stable training
- Monotonic improvement (mostly)

**2. Simplicity**:
- Straightforward implementation
- First-order optimization
- Easy to debug
- Few moving parts

**3. Performance**:
- State-of-the-art on many benchmarks
- Good sample efficiency
- Handles continuous and discrete actions

**4. Scalability**:
- Parallelizable (multiple workers)
- Works with large networks
- GPU-friendly

**5. Versatility**:
- Robotics
- Games
- RLHF (ChatGPT, Claude)
- Continuous control
- Discrete control

### Real-World Applications

**OpenAI**:
- Dota 2 bot (OpenAI Five)
- ChatGPT/GPT-4 RLHF
- Robotics (Dactyl cube manipulation)

**DeepMind**:
- AlphaStar (StarCraft II)
- Various robotics projects

**Industry**:
- Autonomous vehicles
- Recommendation systems
- Industrial control

**2025 status**: PPO is default choice for policy gradient problems

### When NOT to Use PPO

**Use DQN instead if**:
- Discrete actions + sample efficiency critical
- Need off-policy (replay buffer)
- Deterministic policy okay

**Use SAC instead if**:
- Continuous control
- Maximum sample efficiency needed
- Off-policy benefits important

**Use TRPO if**:
- Need theoretical guarantees
- Willing to pay computational cost
- Maximum stability required

**But usually**: PPO is the answer!

## Summary

### Key Concepts

✅ **Trust Regions** - Constrain policy updates for stability  
✅ **TRPO** - KL-constrained policy optimization  
✅ **PPO-Clip** - Clipped surrogate objective  
✅ **Importance Sampling** - Reuse old data  
✅ **GAE** - Generalized advantage estimation  
✅ **Multiple Epochs** - Better sample efficiency  

### Critical Equations

**PPO objective**:
$$L^{CLIP}(\theta) = \mathbb{E}_t [\min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t)]$$

**Probability ratio**:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

**Full loss**:
$$L = -L^{CLIP} + c_1 L^{VF} - c_2 H$$

### Implementation Checklist

**Essential**:
- ✓ Clipped objective with ε=0.2
- ✓ GAE for advantage estimation
- ✓ Advantage normalization
- ✓ Multiple epochs (K=10)
- ✓ Gradient clipping

**Important**:
- ✓ Value function loss
- ✓ Entropy bonus
- ✓ Learning rate annealing
- ✓ Minibatch updates

**Optional**:
- Value clipping
- Learning rate scheduling
- Observation normalization

### Practical Recommendations

**Start with OpenAI baselines values**:
- They work surprisingly well!
- ε=0.2, lr=3e-4, K=10
- GAE λ=0.95, γ=0.99

**Tune if needed**:
- Increase K for better sample efficiency
- Decrease ε for more stable updates
- Adjust learning rate if not converging

**Monitor**:
- Policy entropy (should decrease slowly)
- KL divergence (should stay small)
- Explained variance (critic quality)
- Clip fraction (how often clipping active)

### What's Next

**Lesson 9b**: Implement PPO from scratch
- Complete working implementation
- Train on CartPole and continuous control
- Compare with A2C

**Lesson 10**: Continuous Control (DDPG, TD3, SAC)
- Off-policy actor-critic
- Deterministic policies
- Maximum sample efficiency

**Lesson 16**: RLHF
- PPO for language models
- Reward modeling
- ChatGPT-style training

---

**Lesson 9a Complete!** 🎉

You now understand PPO, the most important RL algorithm in 2025!